In [12]:
# ============================================================
# Midland County RRC Production Analysis
# Recreates cleaned workbook and figures from 5 raw RRC CSVs
# ============================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1. Set file paths
# ------------------------------------------------------------

# Your raw CSV files are currently in Downloads
RAW_DIR = Path(r"C:\Users\joelu\Downloads")

# Your project output folders are in Documents
BASE_DIR = Path(r"C:\Users\joelu\Documents")

CLEAN_DIR = BASE_DIR / "data" / "cleaned"
FIGURE_DIR = BASE_DIR / "figures"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DISTRICT_FILE = RAW_DIR / "ProductionDataQuery_District_ReportCsv.csv"
FIELD_FILE = RAW_DIR / "ProductionDataQuery_Field_ReportCsv.csv"
LEASE_FILE = RAW_DIR / "ProductionDataQuery_Lease_ReportCsv.csv"
MONTHLY_FILE = RAW_DIR / "ProductionDataQuery_monthlyTotals_ReportCsv.csv"
OPERATOR_FILE = RAW_DIR / "ProductionDataQuery_Operator_ReportCsv.csv"

OUTPUT_EXCEL = CLEAN_DIR / "Midland_County_RRC_Production_Analysis.xlsx"


# ------------------------------------------------------------
# 2. Check that all required files exist
# ------------------------------------------------------------

required_files = [
    DISTRICT_FILE,
    FIELD_FILE,
    LEASE_FILE,
    MONTHLY_FILE,
    OPERATOR_FILE
]

print("Checking for raw files in:", RAW_DIR)
print()

missing_files = []

for file in required_files:
    if file.exists():
        print("FOUND:", file.name)
    else:
        print("MISSING:", file.name)
        missing_files.append(file.name)

if missing_files:
    raise FileNotFoundError(
        "These required files were not found in Downloads:\n"
        + "\n".join(missing_files)
        + "\n\nMake sure the file names match exactly."
    )

print()
print("All 5 raw files found. Continuing analysis...")


# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------

def read_rrc_csv(file_path: Path) -> pd.DataFrame:
    """
    Reads a Texas RRC CSV export.

    RRC Production Data Query CSV exports usually have search criteria
    above the real table header. For these Midland County files,
    the real header starts after 7 skipped rows.
    """
    df = pd.read_csv(file_path, skiprows=7, dtype=str)
    return df


def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Removes extra spaces from column names.
    """
    df = df.copy()
    df.columns = df.columns.str.strip()
    return df


def remove_total_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Removes rows where the first column says TOTAL.
    """
    df = df.copy()
    first_col = df.columns[0]
    df = df[df[first_col].astype(str).str.strip().str.upper() != "TOTAL"]
    return df


def convert_numeric_columns(df: pd.DataFrame, numeric_cols: list) -> pd.DataFrame:
    """
    Converts selected production columns to numbers.
    Removes commas and turns blanks/errors into 0.
    """
    df = df.copy()

    for col in numeric_cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace(",", "", regex=False)
                .str.strip()
            )
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    return df


def add_gas_total_and_gor(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds:
    - Gas Total (MCF)
    - GOR (MCF/BBL)
    """
    df = df.copy()

    if "Casinghead (MCF)" not in df.columns:
        df["Casinghead (MCF)"] = 0

    if "GW Gas (MCF)" not in df.columns:
        df["GW Gas (MCF)"] = 0

    if "Oil (BBL)" not in df.columns:
        df["Oil (BBL)"] = 0

    df["Gas Total (MCF)"] = df["Casinghead (MCF)"] + df["GW Gas (MCF)"]

    df["GOR (MCF/BBL)"] = df.apply(
        lambda row: row["Gas Total (MCF)"] / row["Oil (BBL)"]
        if row["Oil (BBL)"] > 0
        else 0,
        axis=1
    )

    return df


def add_oil_share(df: pd.DataFrame, total_oil: float) -> pd.DataFrame:
    """
    Adds each row's share of total county oil production.
    """
    df = df.copy()

    if total_oil > 0:
        df["Share of County Oil"] = df["Oil (BBL)"] / total_oil
    else:
        df["Share of County Oil"] = 0

    return df


def rank_by_oil(df: pd.DataFrame) -> pd.DataFrame:
    """
    Sorts by oil production and adds a rank column.
    """
    df = df.copy()
    df = df.sort_values("Oil (BBL)", ascending=False).reset_index(drop=True)
    df.insert(0, "Rank", range(1, len(df) + 1))
    return df


def clean_text_column(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """
    Strips spaces from a text column if it exists.
    """
    df = df.copy()

    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

    return df


# ------------------------------------------------------------
# 4. Load raw data
# ------------------------------------------------------------

district_raw = read_rrc_csv(DISTRICT_FILE)
fields_raw = read_rrc_csv(FIELD_FILE)
leases_raw = read_rrc_csv(LEASE_FILE)
monthly_raw = read_rrc_csv(MONTHLY_FILE)
operators_raw = read_rrc_csv(OPERATOR_FILE)

print()
print("Raw files loaded successfully.")
print("District rows:", len(district_raw))
print("Field rows:", len(fields_raw))
print("Lease rows:", len(leases_raw))
print("Monthly rows:", len(monthly_raw))
print("Operator rows:", len(operators_raw))


# ------------------------------------------------------------
# 5. Clean basic formatting
# ------------------------------------------------------------

district = clean_column_names(district_raw)
fields = clean_column_names(fields_raw)
leases = clean_column_names(leases_raw)
monthly = clean_column_names(monthly_raw)
operators = clean_column_names(operators_raw)

numeric_cols = [
    "Oil (BBL)",
    "Casinghead (MCF)",
    "GW Gas (MCF)",
    "Condensate (BBL)"
]

district = convert_numeric_columns(district, numeric_cols)
fields = convert_numeric_columns(fields, numeric_cols)
leases = convert_numeric_columns(leases, numeric_cols)
monthly = convert_numeric_columns(monthly, numeric_cols)
operators = convert_numeric_columns(operators, numeric_cols)


# ------------------------------------------------------------
# 6. Remove total rows
# ------------------------------------------------------------

district_clean = district.copy()
fields = remove_total_rows(fields)
leases = remove_total_rows(leases)
monthly = remove_total_rows(monthly)
operators = remove_total_rows(operators)


# ------------------------------------------------------------
# 7. Clean text columns
# ------------------------------------------------------------

for col in ["Field Name", "Field No."]:
    fields = clean_text_column(fields, col)

for col in ["Operator Name", "Operator No."]:
    operators = clean_text_column(operators, col)

for col in ["Lease Name", "Lease No.", "Well No.", "District No.", "Field Name", "Operator Name"]:
    leases = clean_text_column(leases, col)


# ------------------------------------------------------------
# 8. Clean monthly dates
# ------------------------------------------------------------

monthly["Date"] = monthly["Date"].astype(str).str.strip()

monthly["Date"] = pd.to_datetime(
    monthly["Date"],
    format="%b %Y",
    errors="coerce"
)

monthly = monthly.dropna(subset=["Date"]).reset_index(drop=True)
monthly["Year"] = monthly["Date"].dt.year

monthly = add_gas_total_and_gor(monthly)

monthly = monthly[
    [
        "Date",
        "Year",
        "Oil (BBL)",
        "Casinghead (MCF)",
        "GW Gas (MCF)",
        "Gas Total (MCF)",
        "Condensate (BBL)",
        "GOR (MCF/BBL)"
    ]
]


# ------------------------------------------------------------
# 9. Annual summary
# ------------------------------------------------------------

annual_summary = (
    monthly
    .groupby("Year", as_index=False)
    .agg(
        **{
            "Months in Dataset": ("Date", "count"),
            "Oil (BBL)": ("Oil (BBL)", "sum"),
            "Gas Total (MCF)": ("Gas Total (MCF)", "sum"),
            "Condensate (BBL)": ("Condensate (BBL)", "sum"),
        }
    )
)

annual_summary["GOR (MCF/BBL)"] = annual_summary.apply(
    lambda row: row["Gas Total (MCF)"] / row["Oil (BBL)"]
    if row["Oil (BBL)"] > 0
    else 0,
    axis=1
)


# ------------------------------------------------------------
# 10. Field analysis
# ------------------------------------------------------------

fields = add_gas_total_and_gor(fields)

field_total_oil = fields["Oil (BBL)"].sum()
field_total_gas = fields["Gas Total (MCF)"].sum()
field_total_condensate = fields["Condensate (BBL)"].sum()

fields = add_oil_share(fields, total_oil)
fields_ranked = rank_by_oil(fields)

field_columns = [
    "Rank",
    "Field Name",
    "Field No.",
    "Oil (BBL)",
    "Gas Total (MCF)",
    "Condensate (BBL)",
    "GOR (MCF/BBL)",
    "Share of County Oil"
]

field_report = fields_ranked[[col for col in field_columns if col in fields_ranked.columns]].head(20)

all_fields_columns = [
    "Field Name",
    "Field No.",
    "Oil (BBL)",
    "Gas Total (MCF)",
    "Condensate (BBL)",
    "GOR (MCF/BBL)",
    "Share of County Oil"
]

all_fields = fields_ranked[[col for col in all_fields_columns if col in fields_ranked.columns]].copy()


# ------------------------------------------------------------
# 11. Operator analysis
# ------------------------------------------------------------

operators = add_gas_total_and_gor(operators)
operators = add_oil_share(operators, total_oil)
operators_ranked = rank_by_oil(operators)

operator_columns = [
    "Rank",
    "Operator Name",
    "Operator No.",
    "Oil (BBL)",
    "Gas Total (MCF)",
    "Condensate (BBL)",
    "GOR (MCF/BBL)",
    "Share of County Oil"
]

operator_report = operators_ranked[[col for col in operator_columns if col in operators_ranked.columns]].head(20)


# ------------------------------------------------------------
# 12. Lease analysis
# ------------------------------------------------------------

# First add Gas Total and GOR columns
leases = add_gas_total_and_gor(leases)

# Now calculate lease totals
lease_total_oil = leases["Oil (BBL)"].sum()
lease_total_gas = leases["Gas Total (MCF)"].sum()
lease_total_condensate = leases["Condensate (BBL)"].sum()

# Add share of lease dataset oil, not field dataset oil
leases = add_oil_share(leases, lease_total_oil)

# Rank leases by oil production
leases_ranked = rank_by_oil(leases)

lease_columns = [
    "Rank",
    "Lease Name",
    "Lease No.",
    "Oil (BBL)",
    "Gas Total (MCF)",
    "Condensate (BBL)",
    "GOR (MCF/BBL)",
    "Share of County Oil"
]

lease_report = leases_ranked[[col for col in lease_columns if col in leases_ranked.columns]].head(20)

# ------------------------------------------------------------
# 13. Lease concentration analysis
# ------------------------------------------------------------

import math

lease_count = len(leases_ranked)
top_10_percent_count = math.ceil(lease_count * 0.10)

top_10_percent_oil = leases_ranked.head(top_10_percent_count)["Oil (BBL)"].sum()
top_10_percent_share = top_10_percent_oil / lease_total_oil if lease_total_oil > 0 else 0

leases_ranked["Cumulative Oil (BBL)"] = leases_ranked["Oil (BBL)"].cumsum()
leases_ranked["Cumulative Oil Share"] = leases_ranked["Cumulative Oil (BBL)"] / lease_total_oil
leases_ranked["Cumulative Lease Share"] = (leases_ranked.index + 1) / lease_count


# ------------------------------------------------------------
# 14. Executive summary values
# ------------------------------------------------------------

oil_2020 = annual_summary.loc[annual_summary["Year"] == 2020, "Oil (BBL)"].iloc[0]
oil_2025 = annual_summary.loc[annual_summary["Year"] == 2025, "Oil (BBL)"].iloc[0]

gas_2020 = annual_summary.loc[annual_summary["Year"] == 2020, "Gas Total (MCF)"].iloc[0]
gas_2025 = annual_summary.loc[annual_summary["Year"] == 2025, "Gas Total (MCF)"].iloc[0]

oil_growth_2020_2025 = (oil_2025 - oil_2020) / oil_2020
gas_growth_2020_2025 = (gas_2025 - gas_2020) / gas_2020

average_gor = total_gas / total_oil if total_oil > 0 else 0

top_field = field_report.iloc[0]["Field Name"]
top_operator = operator_report.iloc[0]["Operator Name"]
top_lease = lease_report.iloc[0]["Lease Name"]

executive_summary = pd.DataFrame(
    {
        "Metric": [
            "Data source",
            "County",
            "Date range",
            "Total oil (BBL)",
            "Total gas (MCF)",
            "Total condensate (BBL)",
            "Average GOR (MCF/BBL)",
            "2020 to 2025 oil growth",
            "2020 to 2025 gas growth",
            "Top field by oil production",
            "Top operator by oil production",
            "Top lease by oil production",
            "Number of leases analyzed",
            "Top 10% lease count",
            "Oil share from top 10% of leases"
        ],
        "Value": [
            "Texas Railroad Commission Production Data Query",
            "Midland County, Texas",
            "Jan 2020 - Mar 2026",
            total_oil,
            total_gas,
            total_condensate,
            average_gor,
            oil_growth_2020_2025,
            gas_growth_2020_2025,
            top_field,
            top_operator,
            top_lease,
            lease_count,
            top_10_percent_count,
            top_10_percent_share
        ]
    }
)

source_notes = pd.DataFrame(
    {
        "Item": [
            "Source",
            "County",
            "Date range",
            "Raw files",
            "Important limitation"
        ],
        "Description": [
            "Texas Railroad Commission Production Data Query CSV exports",
            "Midland County, Texas",
            "Jan 2020 - Mar 2026",
            "District, Field, Lease, Monthly Totals, and Operator reports",
            "Aggregated RRC query exports do not include completion dates, lateral lengths, proppant loading, or detailed well-level decline data."
        ]
    }
)


# ------------------------------------------------------------
# 15. Create graph images
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))
plt.plot(monthly["Date"], monthly["Oil (BBL)"], linewidth=2)
plt.title("Midland County Monthly Oil Production")
plt.xlabel("Date")
plt.ylabel("Oil Production (BBL)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "monthly_oil.png", dpi=300)
plt.close()


plt.figure(figsize=(10, 6))
plt.plot(monthly["Date"], monthly["Gas Total (MCF)"], linewidth=2)
plt.title("Midland County Monthly Gas Production")
plt.xlabel("Date")
plt.ylabel("Gas Production (MCF)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "monthly_gas.png", dpi=300)
plt.close()


plt.figure(figsize=(10, 6))
plt.plot(monthly["Date"], monthly["GOR (MCF/BBL)"], linewidth=2)
plt.title("Midland County Gas-Oil Ratio Trend")
plt.xlabel("Date")
plt.ylabel("GOR (MCF/BBL)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "gor_trend.png", dpi=300)
plt.close()


top_operators_plot = operator_report.head(10).sort_values("Oil (BBL)", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(top_operators_plot["Operator Name"], top_operators_plot["Oil (BBL)"])
plt.title("Top 10 Midland County Operators by Oil Production")
plt.xlabel("Oil Production (BBL)")
plt.ylabel("Operator")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "top_operators_oil.png", dpi=300)
plt.close()


top_fields_plot = field_report.head(10).sort_values("Oil (BBL)", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(top_fields_plot["Field Name"], top_fields_plot["Oil (BBL)"])
plt.title("Top 10 Midland County Fields by Oil Production")
plt.xlabel("Oil Production (BBL)")
plt.ylabel("Field")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "top_fields_oil.png", dpi=300)
plt.close()


plt.figure(figsize=(10, 6))
plt.plot(
    leases_ranked["Cumulative Lease Share"] * 100,
    leases_ranked["Cumulative Oil Share"] * 100,
    linewidth=2
)
plt.axvline(10, linestyle="--", linewidth=1)
plt.axhline(top_10_percent_share * 100, linestyle="--", linewidth=1)
plt.title("Midland County Lease Production Concentration")
plt.xlabel("Cumulative Share of Leases (%)")
plt.ylabel("Cumulative Share of Oil Production (%)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "lease_concentration.png", dpi=300)
plt.close()


# ------------------------------------------------------------
# 16. Export cleaned Excel workbook
# ------------------------------------------------------------

with pd.ExcelWriter(OUTPUT_EXCEL, engine="xlsxwriter") as writer:
    executive_summary.to_excel(writer, sheet_name="Executive Summary", index=False)
    monthly.to_excel(writer, sheet_name="Monthly Trends", index=False)
    annual_summary.to_excel(writer, sheet_name="Annual Summary", index=False)
    field_report.to_excel(writer, sheet_name="Field Report", index=False)
    operator_report.to_excel(writer, sheet_name="Operator Report", index=False)
    lease_report.to_excel(writer, sheet_name="Lease Report", index=False)
    all_fields.to_excel(writer, sheet_name="All Fields", index=False)
    source_notes.to_excel(writer, sheet_name="Source Notes", index=False)

    workbook = writer.book

    header_format = workbook.add_format(
        {
            "bold": True,
            "bg_color": "#D9EAF7",
            "border": 1,
            "text_wrap": True
        }
    )

    number_format = workbook.add_format({"num_format": "#,##0"})
    decimal_format = workbook.add_format({"num_format": "0.000"})
    percent_format = workbook.add_format({"num_format": "0.0%"})
    date_format = workbook.add_format({"num_format": "mmm yyyy"})

    sheet_dataframes = {
        "Executive Summary": executive_summary,
        "Monthly Trends": monthly,
        "Annual Summary": annual_summary,
        "Field Report": field_report,
        "Operator Report": operator_report,
        "Lease Report": lease_report,
        "All Fields": all_fields,
        "Source Notes": source_notes
    }

    for sheet_name, df in sheet_dataframes.items():
        worksheet = writer.sheets[sheet_name]
        worksheet.freeze_panes(1, 0)

        for col_num, col_name in enumerate(df.columns):
            worksheet.write(0, col_num, col_name, header_format)

        for i, col in enumerate(df.columns):
            max_length = max(
                df[col].astype(str).map(len).max() if len(df) > 0 else 0,
                len(str(col))
            )
            width = min(max(max_length + 2, 12), 40)

            if col == "Date":
                worksheet.set_column(i, i, 14, date_format)
            elif "Share" in col or "growth" in col.lower():
                worksheet.set_column(i, i, 18, percent_format)
            elif "GOR" in col:
                worksheet.set_column(i, i, 18, decimal_format)
            elif any(word in col for word in ["Oil", "Gas", "Condensate", "count", "Total", "Number"]):
                worksheet.set_column(i, i, 18, number_format)
            else:
                worksheet.set_column(i, i, width)

    charts_sheet = workbook.add_worksheet("Charts")
    charts_sheet.write("A1", "Charts generated as PNG files in the figures folder.")
    charts_sheet.write("A3", "monthly_oil.png")
    charts_sheet.write("A4", "monthly_gas.png")
    charts_sheet.write("A5", "gor_trend.png")
    charts_sheet.write("A6", "top_operators_oil.png")
    charts_sheet.write("A7", "top_fields_oil.png")
    charts_sheet.write("A8", "lease_concentration.png")


# ------------------------------------------------------------
# 17. Final output
# ------------------------------------------------------------

print()
print("Analysis complete.")
print("Cleaned workbook saved to:")
print(OUTPUT_EXCEL)
print()
print("Figures saved to:")
print(FIGURE_DIR)
print()
print("Key findings:")
print(f"2020 to 2025 oil growth: {oil_growth_2020_2025:.1%}")
print(f"2020 to 2025 gas growth: {gas_growth_2020_2025:.1%}")
print(f"Top field: {top_field}")
print(f"Top operator: {top_operator}")
print(f"Number of leases analyzed: {lease_count:,}")
print(f"Top 10% of leases account for: {top_10_percent_share:.1%} of oil production")

Checking for raw files in: C:\Users\joelu\Downloads

FOUND: ProductionDataQuery_District_ReportCsv.csv
FOUND: ProductionDataQuery_Field_ReportCsv.csv
FOUND: ProductionDataQuery_Lease_ReportCsv.csv
FOUND: ProductionDataQuery_monthlyTotals_ReportCsv.csv
FOUND: ProductionDataQuery_Operator_ReportCsv.csv

All 5 raw files found. Continuing analysis...

Raw files loaded successfully.
District rows: 3
Field rows: 95
Lease rows: 5675
Monthly rows: 76
Operator rows: 186

Analysis complete.
Cleaned workbook saved to:
C:\Users\joelu\Documents\data\cleaned\Midland_County_RRC_Production_Analysis.xlsx

Figures saved to:
C:\Users\joelu\Documents\figures

Key findings:
2020 to 2025 oil growth: 26.1%
2020 to 2025 gas growth: 78.7%
Top field: SPRABERRY (TREND AREA)
Top operator: PIONEER NATURAL RES. USA, INC.
Number of leases analyzed: 5,674
Top 10% of leases account for: 58.1% of oil production
